# SIC 2026 · Inteligencia Artificial
## Clase 6: Gradio a fondo, Kaggle y un proyecto completo publicado en Hugging Face

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

En la Clase 4 hicimos una primera app con Gradio. Hoy vamos más profundo y recorremos todo lo que van a necesitar para su proyecto capstone:

1. **Gradio a fondo:** componentes, varias entradas y salidas, imágenes, gráficos y diseño con `Blocks`.
2. **Kaggle:** qué es, cómo elegir un dataset y cómo descargarlo desde Colab.
3. **Proyecto completo:** predecir qué clientes de una compañía de telecomunicaciones se van a ir, con un dataset de Kaggle.
4. **Publicar en Hugging Face Spaces:** subir la app paso a paso para que quede con un link permanente.
5. **Desafío:** repetir el proceso con un dataset de Kaggle elegido por ustedes.

| Hora | Actividad |
|---|---|
| 08:00 a 08:10 | Inicio |
| 08:10 a 09:10 | Parte 1. Gradio a fondo |
| 09:10 a 09:40 | Parte 2. Kaggle |
| 09:40 a 09:50 | Pausa |
| 09:50 a 11:10 | Parte 3. Proyecto completo: fuga de clientes |
| 11:10 a 11:40 | Parte 4. Publicar en Hugging Face Spaces |
| 11:40 a 12:00 | Parte 5. Desafío y cierre |

Como en las clases anteriores: cada parte tiene definiciones, código que funciona, ejercicios con verificación y soluciones al final.

In [ ]:
!pip install -q gradio kagglehub huggingface_hub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
import warnings
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

print("Gradio", gr.__version__)

---
# Parte 1. Gradio a fondo

> **Definición. Gradio:** librería de Python para construir interfaces web para funciones y modelos. Lo que hace por debajo es simple: muestra campos de entrada, llama a **una función de Python** con lo que la persona ingresó y muestra lo que la función devuelve.

La idea más importante de esta parte:

> **Una app de Gradio es una función de Python con una interfaz encima.** Si la función funciona sola, la app funciona.

Gradio tiene dos formas de construir apps:

| Forma | Cuándo usarla |
|---|---|
| `gr.Interface` | Apps simples: entradas, una función, salidas. Se arma en pocas líneas. |
| `gr.Blocks` | Apps con diseño propio: columnas, pestañas, botones, varias funciones. |

## 1.1 La app más simple

`gr.Interface` recibe tres cosas: la función (`fn`), los componentes de entrada (`inputs`) y los de salida (`outputs`). Los textos `"text"` y `"number"` son atajos para componentes básicos.

In [ ]:
def saludar(nombre):
    return f"Hola, {nombre}. Bienvenida o bienvenido a SIC 2026."

# Primero probamos la función sola
print(saludar("Camila"))

app_saludo = gr.Interface(fn=saludar, inputs="text", outputs="text", flagging_mode="never")
app_saludo.launch()

`flagging_mode="never"` oculta el botón *Flag*, que sirve para guardar ejemplos y que en clases solo distrae.

## 1.2 Componentes más usados

Cada componente tiene parámetros como `label` (el título que ve la persona), `value` (valor inicial) e `info` (texto de ayuda).

| Componente | Para qué | Tipo que llega a la función |
|---|---|---|
| `gr.Textbox` | Texto libre | `str` |
| `gr.Number` | Un número | `float` (o `int` con `precision=0`) |
| `gr.Slider` | Un número dentro de un rango | `float` |
| `gr.Dropdown` | Elegir de una lista | `str` |
| `gr.Radio` | Elegir de pocas opciones, todas visibles | `str` |
| `gr.Checkbox` | Sí o no | `bool` |
| `gr.Image` | Subir o mostrar una imagen | arreglo de NumPy (con `type="numpy"`) |
| `gr.Dataframe` | Mostrar o editar una tabla | `DataFrame` de pandas |
| `gr.Label` | Mostrar clases con probabilidades | se le entrega un `dict` |
| `gr.Plot` | Mostrar un gráfico | se le entrega una figura de matplotlib |
| `gr.Markdown` | Texto con formato | se le entrega un `str` |

## 1.3 Varias entradas y varias salidas: simulador de crédito

Regla clave: **el orden de `inputs` es el orden de los argumentos de la función, y el orden de `outputs` es el orden de lo que la función devuelve.**

Vamos a simular la cuota mensual de un crédito de consumo en pesos chilenos con la fórmula de cuota fija:

$$\text{cuota} = M \cdot \frac{i}{1 - (1 + i)^{-n}}$$

donde $M$ es el monto, $i$ la tasa mensual y $n$ el número de meses.

In [ ]:
def simular_credito(monto, tasa_anual, meses, con_seguro):
    i = tasa_anual / 100 / 12                       # tasa mensual en decimal
    cuota = monto * i / (1 - (1 + i) ** (-meses))
    if con_seguro:
        cuota += monto * 0.0005                     # seguro simple de ejemplo: 0,05% del monto al mes
    total = cuota * meses
    resumen = f"**Pagarás {meses} cuotas.** Los intereses y el seguro suman ${total - monto:,.0f} sobre lo pedido.".replace(",", ".")
    return round(cuota), round(total), resumen

# Prueba de la función sola
print(simular_credito(3_000_000, 24, 36, False))

In [ ]:
app_credito = gr.Interface(
    fn=simular_credito,
    inputs=[
        gr.Number(value=3_000_000, label="Monto del crédito (CLP)", precision=0),
        gr.Slider(5, 45, value=24, step=0.5, label="Tasa de interés anual (%)"),
        gr.Slider(6, 60, value=36, step=6, label="Plazo (meses)"),
        gr.Checkbox(value=False, label="Incluir seguro de desgravamen"),
    ],
    outputs=[
        gr.Number(label="Cuota mensual (CLP)"),
        gr.Number(label="Total a pagar (CLP)"),
        gr.Markdown(),
    ],
    title="Simulador de crédito de consumo",
    description="Ejemplo educativo con cuota fija. No reemplaza una simulación oficial del banco.",
    examples=[[1_000_000, 18, 12, False], [5_000_000, 28, 48, True]],
    flagging_mode="never",
)
app_credito.launch()

## 1.4 Imágenes: la imagen es un arreglo de NumPy

Con `gr.Image(type="numpy")`, la función recibe la imagen como un arreglo de forma `(alto, ancho, 3)`: tres canales de color (rojo, verde y azul) con valores de 0 a 255. Todo lo que vimos en la Clase 2 con NumPy sirve para procesarla.

In [ ]:
def procesar_imagen(imagen, efecto):
    if imagen is None:
        return None
    img = imagen.astype(float)
    if efecto == "Escala de grises":
        gris = img @ np.array([0.299, 0.587, 0.114])    # promedio ponderado de los 3 canales
        img = np.stack([gris] * 3, axis=-1)
    elif efecto == "Negativo":
        img = 255 - img
    elif efecto == "Más brillo":
        img = np.clip(img * 1.4, 0, 255)
    return img.astype(np.uint8)

# Prueba con una imagen sintética de 2 x 2 pixeles
prueba = np.array([[[255, 0, 0], [0, 255, 0]], [[0, 0, 255], [255, 255, 255]]], dtype=np.uint8)
print(procesar_imagen(prueba, "Negativo")[0, 0])   # el rojo puro pasa a cian

In [ ]:
app_imagen = gr.Interface(
    fn=procesar_imagen,
    inputs=[gr.Image(type="numpy", label="Sube una imagen"),
            gr.Radio(["Escala de grises", "Negativo", "Más brillo"], value="Escala de grises", label="Efecto")],
    outputs=gr.Image(label="Resultado"),
    title="Filtros de imagen con NumPy",
    flagging_mode="never",
)
app_imagen.launch()

## 1.5 Diseño propio con `gr.Blocks`

`gr.Interface` decide el diseño por ustedes. Con `gr.Blocks` lo deciden ustedes:

- `gr.Row()` pone componentes uno al lado del otro; `gr.Column()` uno debajo del otro.
- `gr.Tab("nombre")` crea pestañas.
- `gr.Button` crea un botón, y `boton.click(fn, inputs, outputs)` conecta el botón con una función.
- Todo se escribe dentro de un bloque `with gr.Blocks() as app:`.

Rehacemos el simulador con dos pestañas: la cuota y un gráfico de cómo baja la deuda mes a mes.

In [ ]:
def grafico_deuda(monto, tasa_anual, meses):
    i = tasa_anual / 100 / 12
    cuota = monto * i / (1 - (1 + i) ** (-meses))
    saldo, saldos = monto, []
    for _ in range(int(meses)):
        saldo = saldo * (1 + i) - cuota              # se suma el interés del mes y se resta la cuota
        saldos.append(max(saldo, 0))
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(range(1, int(meses) + 1), np.array(saldos) / 1e6, color="#1428A0")
    ax.set_xlabel("Mes"); ax.set_ylabel("Deuda pendiente (millones CLP)"); ax.set_title("Cómo baja la deuda")
    ax.grid(alpha=0.3)
    return fig

with gr.Blocks(title="Simulador de crédito") as app_blocks:
    gr.Markdown("# Simulador de crédito de consumo")
    with gr.Row():
        with gr.Column():
            monto = gr.Number(value=3_000_000, label="Monto (CLP)", precision=0)
            tasa = gr.Slider(5, 45, value=24, step=0.5, label="Tasa anual (%)")
            meses = gr.Slider(6, 60, value=36, step=6, label="Plazo (meses)")
            seguro = gr.Checkbox(label="Incluir seguro")
            boton = gr.Button("Calcular", variant="primary")
        with gr.Column():
            with gr.Tab("Cuota"):
                cuota_out = gr.Number(label="Cuota mensual (CLP)")
                total_out = gr.Number(label="Total a pagar (CLP)")
                resumen_out = gr.Markdown()
            with gr.Tab("Gráfico"):
                grafico_out = gr.Plot()

    # Un botón puede disparar varias funciones
    boton.click(simular_credito, inputs=[monto, tasa, meses, seguro], outputs=[cuota_out, total_out, resumen_out])
    boton.click(grafico_deuda, inputs=[monto, tasa, meses], outputs=grafico_out)

app_blocks.launch()

### ¿Cuándo usar cada uno?

| Necesito... | Uso |
|---|---|
| Mostrar un modelo rápido, con pocas entradas | `gr.Interface` |
| Ordenar entradas en columnas, agregar pestañas o explicaciones | `gr.Blocks` |
| Que un botón haga varias cosas o que haya varias funciones | `gr.Blocks` |

Para su proyecto, `gr.Blocks` suele verse mejor y deja espacio para explicar qué hace el modelo y sus límites.

## Ejercicios de Gradio

### Ejercicio 1
Escribe la función `convertir(monto_clp, moneda)` que convierta pesos chilenos a otra moneda usando el diccionario `TASAS` (cuántos pesos vale una unidad de cada moneda). Debe devolver el resultado **redondeado a 2 decimales**. Luego crea `app_conversor` con `gr.Interface`, usando un `gr.Number` y un `gr.Dropdown` como entradas.

In [ ]:
TASAS = {"USD": 950.0, "EUR": 1030.0, "UF": 39500.0}   # valores de ejemplo, no actualizados

def convertir(monto_clp, moneda):
    return None  # TU CÓDIGO AQUÍ

app_conversor = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: convertir(95000, "USD") == 100.0 and convertir(79000, "UF") == 2.0
        and isinstance(app_conversor, gr.Interface))

### Ejercicio 2
Agrega un efecto `"Espejo"` a `procesar_imagen` que voltee la imagen de izquierda a derecha. Pista: en un arreglo de forma `(alto, ancho, 3)`, invertir el eje del ancho es `img[:, ::-1]`. Redefine la función completa con el nuevo efecto.

In [ ]:
# TU CÓDIGO AQUÍ: redefine procesar_imagen con el efecto "Espejo"

In [ ]:
revisar(lambda: np.array_equal(procesar_imagen(prueba, "Espejo"), prueba[:, ::-1]))

---
# Parte 2. Kaggle

> **Definición. Kaggle:** plataforma de Google para ciencia de datos. Tiene tres cosas que nos sirven: **datasets** públicos (más de 400.000), **notebooks** de otras personas que los analizan, y **competencias** con premios.

Para el capstone, Kaggle es la forma más rápida de encontrar datos. Pero un dataset de Kaggle no está automáticamente bien: hay que revisarlo.

## 2.1 Crear una cuenta

1. Entra a [kaggle.com](https://www.kaggle.com) y regístrate con tu cuenta de Google.
2. Verifica tu cuenta con tu número de teléfono (*Settings > Phone verification*). Algunas funciones lo exigen.

## 2.2 Elegir un buen dataset

En [kaggle.com/datasets](https://www.kaggle.com/datasets), busquen por tema y filtren por tipo de archivo (CSV es lo más simple). Antes de elegir, revisen en la página del dataset:

| Qué mirar | Dónde | Por qué importa |
|---|---|---|
| **Usability** | Puntaje de 0 a 10, arriba a la derecha | Sobre 8 indica buena documentación |
| **Descripción** (*Data Card*) | Pestaña principal | Explica de dónde salen los datos y qué significa cada columna |
| **Licencia** | Barra lateral | Indica si se pueden usar y compartir (CC0 y CC BY son las más abiertas) |
| **Tamaño y número de filas** | Pestaña *Data* | Muy pocos datos dan modelos inestables; muchos pueden no caber en Colab |
| **Notebooks** | Pestaña *Code* | Ver cómo otros lo analizaron y qué resultados obtuvieron |
| **Fecha de actualización** | Barra lateral | Datos muy antiguos pueden no representar la realidad actual |

## 2.3 Tres formas de descargar un dataset

| Forma | Cuándo usarla | Requiere token |
|---|---|---|
| A. Descarga manual | Para mirar rápido; subir a Colab con el panel de archivos | No |
| B. `kagglehub` | Datasets públicos desde Colab, en una línea | No (para datasets públicos) |
| C. API de Kaggle (`kaggle.json`) | Competencias, datasets que exigen aceptar condiciones, uso frecuente | Sí |

Cada dataset tiene un **identificador** con la forma `usuario/nombre-del-dataset`, que aparece en la URL. Por ejemplo, en `kaggle.com/datasets/blastchar/telco-customer-churn` el identificador es `blastchar/telco-customer-churn`.

### Forma B: `kagglehub` (la que usaremos)

`kagglehub.dataset_download` descarga el dataset y devuelve la carpeta donde quedó. Si la red de Kaggle falla, usamos una copia de respaldo del mismo archivo.

In [ ]:
import os
import kagglehub

def descargar_telco():
    """Descarga el dataset de fuga de clientes desde Kaggle; si falla, usa una copia pública del mismo archivo."""
    try:
        carpeta = kagglehub.dataset_download("blastchar/telco-customer-churn")
        print("Descargado desde Kaggle en:", carpeta)
        print("Archivos:", os.listdir(carpeta))
        return pd.read_csv(os.path.join(carpeta, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))
    except Exception as e:
        print("No se pudo descargar desde Kaggle (", type(e).__name__, "). Uso la copia de respaldo.")
        return pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

clientes = descargar_telco()
print("Forma:", clientes.shape)

### Forma C: API de Kaggle con token (referencia)

La necesitarán si su proyecto usa una **competencia** de Kaggle o un dataset que pide aceptar reglas.

1. En Kaggle, ve a tu perfil > *Settings* > *API* > **Create New Token**. Se descarga un archivo `kaggle.json` con tu usuario y tu clave.
2. **No lo compartas ni lo subas a GitHub:** es como una contraseña.
3. En Colab, guárdalo en el panel de **secretos** (ícono de llave a la izquierda) con los nombres `KAGGLE_USERNAME` y `KAGGLE_KEY`, y activa el acceso del notebook.
4. Usa la API desde el notebook (celda de referencia, no hace falta ejecutarla hoy):

```python
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!kaggle datasets download -d blastchar/telco-customer-churn --unzip -p datos/
# Para una competencia (antes hay que aceptar sus reglas en la web):
# !kaggle competitions download -c spaceship-titanic -p datos/
```

### Ejercicio 3
Escribe el **identificador** del dataset de esta URL en la variable `id_dataset`:
`https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009`

In [ ]:
id_dataset = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: id_dataset == "uciml/red-wine-quality-cortez-et-al-2009")

---
# Parte 3. Proyecto completo: fuga de clientes

Vamos a recorrer el proceso completo con las etapas de CRISP-DM que vio Nicolaj en la Clase 5. Es el mismo flujo de la Clase 4, ahora con datos reales que traen problemas reales.

## Etapa 1. Comprensión del negocio

Una compañía de telecomunicaciones pierde clientes todos los meses. Retener a un cliente cuesta mucho menos que conseguir uno nuevo, así que quiere **identificar a quienes tienen alta probabilidad de irse** para ofrecerles un plan de retención antes de que lo hagan.

> **Definición. Fuga de clientes (churn):** proporción de clientes que dejan de contratar un servicio en un período.

- **Pregunta:** ¿cómo podríamos identificar a los clientes con riesgo de irse, para que el equipo comercial los contacte a tiempo?
- **Variable objetivo:** `Churn` (Yes o No): si el cliente se fue en el último mes.
- **Tipo de problema:** clasificación binaria.
- **Qué error importa más:** no detectar a un cliente que se va (falso negativo) cuesta más que llamar a uno que se iba a quedar (falso positivo). Por eso vamos a mirar el **recall** de la clase "se va", no solo la exactitud.

## Etapa 2. Comprensión de los datos

In [ ]:
clientes.head()

In [ ]:
clientes.info()

Columnas principales: datos del cliente (`gender`, `SeniorCitizen`, `Partner`, `Dependents`), antigüedad en meses (`tenure`), servicios contratados (`InternetService`, `OnlineSecurity`, `TechSupport`, etc.), contrato y pago (`Contract`, `PaperlessBilling`, `PaymentMethod`) y cargos (`MonthlyCharges`, `TotalCharges`, en dólares).

**Algo raro:** `TotalCharges` es de tipo `object` (texto) aunque debería ser un número. Lo investigamos.

In [ ]:
# ¿Qué valores de TotalCharges no se pueden convertir a número?
convertido = pd.to_numeric(clientes["TotalCharges"], errors="coerce")   # lo que no se pueda convertir queda NaN
problemas = clientes.loc[convertido.isna(), ["customerID", "tenure", "TotalCharges"]]
print("Filas con problemas:", len(problemas))
problemas.head()

Son 11 clientes con `TotalCharges` vacío (un espacio en blanco) y todos tienen `tenure = 0`: son clientes nuevos que aún no han pagado nada. Este tipo de detalle aparece en casi todos los datasets reales y por eso siempre hay que revisar `info()`.

Ahora veamos la variable objetivo y algunas relaciones.

In [ ]:
print(clientes["Churn"].value_counts(normalize=True).round(3))

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=clientes, x="Contract", y=clientes["Churn"].eq("Yes"), ax=ejes[0], errorbar=None)
ejes[0].set_title("Tasa de fuga según contrato"); ejes[0].set_ylabel("Proporción que se va")
sns.histplot(data=clientes, x="tenure", hue="Churn", bins=30, ax=ejes[1], multiple="stack")
ejes[1].set_title("Antigüedad (meses) según fuga")
sns.boxplot(data=clientes, x="Churn", y="MonthlyCharges", ax=ejes[2])
ejes[2].set_title("Cargo mensual según fuga")
plt.tight_layout(); plt.show()

**Lectura del EDA:**

- Las clases están **desbalanceadas**: cerca del 27% se va y el 73% se queda. Un modelo que diga siempre "se queda" tendría 73% de exactitud sin servir para nada.
- Los contratos **mes a mes** tienen una fuga mucho mayor que los de uno o dos años.
- Los clientes **nuevos** (poca antigüedad) se van mucho más.
- Quienes se van pagan, en promedio, **cargos mensuales más altos**.

## Etapa 3. Preparación de los datos

Decisiones:

1. Eliminar `customerID`: es un identificador, no aporta información para predecir.
2. Convertir `Churn` a 1 (se va) y 0 (se queda).
3. Elegir las características. Para que la app sea usable, nos quedamos con las 11 más relacionadas con la fuga según el EDA y el conocimiento del negocio. **No usamos `TotalCharges`**: es casi igual a `tenure × MonthlyCharges`, así que no agrega información y obligaría a la persona de la app a calcularlo.

In [ ]:
datos = clientes.drop(columns="customerID").copy()
datos["Churn"] = (datos["Churn"] == "Yes").astype(int)
datos["SeniorCitizen"] = datos["SeniorCitizen"].map({0: "No", 1: "Yes"})   # lo tratamos como categoría

NUMERICAS = ["tenure", "MonthlyCharges"]
CATEGORICAS = ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport",
               "PaperlessBilling", "SeniorCitizen", "Partner", "Dependents"]
CARACTERISTICAS = NUMERICAS + CATEGORICAS

X = datos[CARACTERISTICAS]
y = datos["Churn"]
print("X:", X.shape, "| Tasa de fuga:", round(y.mean(), 3))

### División en entrenamiento y prueba

Con clases desbalanceadas, `stratify=y` es indispensable: asegura que ambos conjuntos tengan cerca de 27% de clientes que se van.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Tasa de fuga en train:", round(y_train.mean(), 3), "| en test:", round(y_test.mean(), 3))

### Pipeline de preprocesamiento

Aquí las categóricas **no tienen orden** (un método de pago no es "mayor" que otro), así que usamos **one-hot encoding**, a diferencia de la Clase 4 donde el corte y el color sí tenían orden.

`handle_unknown="ignore"` evita que el modelo falle si en la app llega una categoría que no vio en el entrenamiento.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

preprocesador = ColumnTransformer([
    ("numericas", StandardScaler(), NUMERICAS),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
])
print("Columnas después de transformar:", preprocesador.fit_transform(X_train).shape[1])

## Etapa 4. Modelado

### Línea base y comparación con validación cruzada

Como el error que más nos importa es no detectar a quien se va, comparamos los modelos con dos métricas:

> **Recall de la clase 1:** de los clientes que realmente se fueron, qué proporción detecta el modelo.
>
> **ROC AUC:** mide qué tan bien el modelo ordena a los clientes por riesgo, sin depender de un umbral. 0,5 es azar y 1 es perfecto.

> **Definición. `class_weight="balanced"`:** le indica al modelo que los errores en la clase minoritaria pesan más. Es una forma simple de enfrentar el desbalance sin modificar los datos.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

candidatos = {
    "Línea base": DummyClassifier(strategy="most_frequent"),
    "Logística": LogisticRegression(max_iter=2000),
    "Logística balanceada": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Árbol (prof. 5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random forest balanceado": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient boosting": GradientBoostingClassifier(random_state=42),
}

filas = []
for nombre, modelo in candidatos.items():
    pipe = Pipeline([("prep", preprocesador), ("modelo", modelo)])
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["accuracy", "recall", "roc_auc"])
    filas.append({"Modelo": nombre, "Exactitud": r["test_accuracy"].mean(),
                  "Recall (se va)": r["test_recall"].mean(), "ROC AUC": r["test_roc_auc"].mean()})

comparacion = pd.DataFrame(filas).set_index("Modelo").round(3)
comparacion

**Lectura de la tabla:**

- La línea base tiene 73% de exactitud y **0% de recall**: no detecta a nadie. Por eso la exactitud sola engaña con clases desbalanceadas.
- Los modelos sin balancear tienen buena exactitud, pero detectan poco más de la mitad de quienes se van.
- Con `class_weight="balanced"`, el recall sube mucho a cambio de algo de exactitud. Para este negocio es un buen intercambio.
- El ROC AUC es parecido entre la logística y los ensambles: ordenan a los clientes por riesgo de forma similar.

### Ajuste de hiperparámetros

Ajustamos la **regresión logística balanceada**: es casi tan buena como los ensambles, es rápida, liviana para la app y sus coeficientes se pueden interpretar. Optimizamos según ROC AUC.

> **Definición. Parámetro C de la regresión logística:** controla la regularización. Un C pequeño restringe más el modelo (más simple); un C grande lo deja ajustarse más a los datos.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe_log = Pipeline([("prep", preprocesador),
                     ("modelo", LogisticRegression(max_iter=2000, class_weight="balanced"))])

busqueda = GridSearchCV(pipe_log, {"modelo__C": [0.01, 0.1, 1, 10]}, cv=cv, scoring="roc_auc", n_jobs=-1)
busqueda.fit(X_train, y_train)
print("Mejor C:", busqueda.best_params_, "| ROC AUC CV:", round(busqueda.best_score_, 3))
modelo_final = busqueda.best_estimator_

## Etapa 5. Evaluación con el conjunto de prueba

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

prob_test = modelo_final.predict_proba(X_test)[:, 1]
pred_test = modelo_final.predict(X_test)

print(f"ROC AUC en prueba: {roc_auc_score(y_test, prob_test):.3f}")
print(classification_report(y_test, pred_test, target_names=["Se queda", "Se va"], digits=3))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["Se queda", "Se va"], cmap="Blues", ax=ejes[0])
ejes[0].set_title("Matriz de confusión")
RocCurveDisplay.from_predictions(y_test, prob_test, ax=ejes[1]); ejes[1].plot([0, 1], [0, 1], "--", color="gray")
ejes[1].set_title("Curva ROC")
plt.tight_layout(); plt.show()

### Elegir el umbral según el negocio

`predict` usa un umbral de 0,5: si la probabilidad de irse es mayor, predice "se va". Pero el umbral es una **decisión de negocio**, no técnica. Si el equipo comercial puede llamar a muchos clientes, conviene un umbral más bajo (detecta más, con más falsas alarmas). Si solo puede llamar a pocos, uno más alto.

In [ ]:
from sklearn.metrics import precision_score, recall_score

tabla_umbrales = pd.DataFrame([{
    "Umbral": u,
    "Clientes a contactar": int((prob_test >= u).sum()),
    "Recall (se va)": recall_score(y_test, prob_test >= u),
    "Precisión (se va)": precision_score(y_test, prob_test >= u),
} for u in [0.3, 0.4, 0.5, 0.6, 0.7]]).round(3)
tabla_umbrales

Con umbral 0,3 se detecta a casi todos los que se van, pero hay que contactar a muchos clientes. Con 0,7 se contacta a menos de la mitad y 6 de cada 10 contactados realmente se iban, pero se escapan más. **Esta tabla es la que mostrarían a la gerencia** para que decida.

### ¿Qué aprendió el modelo?

En la regresión logística, cada coeficiente indica hacia dónde empuja una característica: positivo aumenta el riesgo de fuga, negativo lo disminuye.

In [ ]:
nombres = modelo_final.named_steps["prep"].get_feature_names_out()
coeficientes = pd.Series(modelo_final.named_steps["modelo"].coef_[0], index=nombres).sort_values()
pd.concat([coeficientes.head(6), coeficientes.tail(6)]).plot(kind="barh", figsize=(8, 5), color="#1428A0",
                                                              title="Coeficientes: negativo retiene, positivo aumenta la fuga")
plt.show()

Coincide con el EDA: los contratos largos y la antigüedad retienen; el contrato mes a mes, la fibra óptica y el pago con cheque electrónico aumentan el riesgo.

## Etapa 6. Despliegue: guardar el modelo y crear la app

In [ ]:
import pickle, sklearn

UMBRAL = 0.5
paquete = {
    "modelo": modelo_final,
    "caracteristicas": CARACTERISTICAS,
    "umbral": UMBRAL,
    "roc_auc_prueba": round(roc_auc_score(y_test, prob_test), 3),
    "version_sklearn": sklearn.__version__,
}
os.makedirs("space_fuga", exist_ok=True)
with open("space_fuga/modelo_fuga.pkl", "wb") as f:
    pickle.dump(paquete, f)
print(f"Guardado: {os.path.getsize('space_fuga/modelo_fuga.pkl') / 1024:.0f} KB")

El archivo pesa unos pocos KB: una ventaja de elegir la regresión logística.

### La app con `gr.Blocks`

Guardamos todo en la carpeta `space_fuga/`, porque es exactamente la carpeta que subiremos a Hugging Face en la Parte 4. Escribimos el código de la app como un archivo `app.py` y luego lo ejecutamos aquí para probarlo.

In [ ]:
codigo_app = '''import pickle
import pandas as pd
import gradio as gr

with open("modelo_fuga.pkl", "rb") as f:
    paquete = pickle.load(f)
modelo, UMBRAL = paquete["modelo"], paquete["umbral"]

def predecir(tenure, monthly, contract, internet, payment, security, support, paperless, senior, partner, dependents):
    cliente = pd.DataFrame([{
        "tenure": tenure, "MonthlyCharges": monthly, "Contract": contract, "InternetService": internet,
        "PaymentMethod": payment, "OnlineSecurity": security, "TechSupport": support,
        "PaperlessBilling": paperless, "SeniorCitizen": senior, "Partner": partner, "Dependents": dependents,
    }])
    p = float(modelo.predict_proba(cliente)[0, 1])
    if p >= UMBRAL:
        mensaje = f"**Riesgo alto ({p:.0%}).** Recomendación: contactar con una oferta de retención."
    else:
        mensaje = f"**Riesgo bajo ({p:.0%}).** No requiere acción inmediata."
    return {"Se va": p, "Se queda": 1 - p}, mensaje

SN = ["No", "Yes"]
SERV = ["No", "Yes", "No internet service"]

with gr.Blocks(title="Riesgo de fuga de clientes") as app:
    gr.Markdown("# Riesgo de fuga de clientes\\nIngresa los datos del cliente para estimar la probabilidad de que deje la compañía.")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Contrato y cargos")
            tenure = gr.Slider(0, 72, value=12, step=1, label="Antigüedad (meses)")
            monthly = gr.Slider(18, 120, value=70, step=0.5, label="Cargo mensual (USD)")
            contract = gr.Dropdown(["Month-to-month", "One year", "Two year"], value="Month-to-month", label="Tipo de contrato")
            payment = gr.Dropdown(["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
                                  value="Electronic check", label="Método de pago")
            paperless = gr.Radio(SN, value="Yes", label="Boleta electrónica")
        with gr.Column():
            gr.Markdown("### Servicios y perfil")
            internet = gr.Dropdown(["DSL", "Fiber optic", "No"], value="Fiber optic", label="Servicio de internet")
            security = gr.Radio(SERV, value="No", label="Seguridad en línea")
            support = gr.Radio(SERV, value="No", label="Soporte técnico")
            senior = gr.Radio(SN, value="No", label="Adulto mayor")
            partner = gr.Radio(SN, value="No", label="Tiene pareja")
            dependents = gr.Radio(SN, value="No", label="Tiene dependientes")
    boton = gr.Button("Estimar riesgo", variant="primary")
    with gr.Row():
        etiqueta = gr.Label(label="Probabilidad")
        mensaje = gr.Markdown()
    entradas = [tenure, monthly, contract, internet, payment, security, support, paperless, senior, partner, dependents]
    boton.click(predecir, inputs=entradas, outputs=[etiqueta, mensaje])
    gr.Markdown("*Modelo educativo entrenado con el dataset Telco Customer Churn de Kaggle. No usar para decisiones reales.*")

app.launch()
'''

with open("space_fuga/app.py", "w", encoding="utf-8") as f:
    f.write(codigo_app)
with open("space_fuga/requirements.txt", "w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\npandas\n")
print(os.listdir("space_fuga"))

Para probar la app aquí mismo, nos movemos a la carpeta y ejecutamos el archivo. Así verificamos que `app.py` funciona solo, exactamente como funcionará en Hugging Face.

In [ ]:
%cd space_fuga
exec(open("app.py", encoding="utf-8").read())
%cd ..

## Ejercicios del proyecto

### Ejercicio 4
Calcula la **tasa de fuga por tipo de contrato** (`Contract`) usando `datos` y guárdala en `fuga_por_contrato` (una Series con la proporción de `Churn == 1` en cada tipo de contrato).

In [ ]:
fuga_por_contrato = None  # TU CÓDIGO AQUÍ
fuga_por_contrato

In [ ]:
revisar(lambda: np.allclose(fuga_por_contrato.sort_index().values, datos.groupby("Contract")["Churn"].mean().sort_index().values))

### Ejercicio 5
La gerencia dice que el equipo comercial puede llamar como máximo a **400 clientes** del conjunto de prueba. Encuentra el **umbral más bajo** de la lista `umbrales` con el que la cantidad de clientes a contactar (`prob_test >= umbral`) sea menor o igual a 400. Guárdalo en `umbral_elegido`.

In [ ]:
umbrales = np.round(np.arange(0.30, 0.91, 0.05), 2)
umbral_elegido = None  # TU CÓDIGO AQUÍ
print(umbral_elegido)

In [ ]:
_ok = [u for u in umbrales if (prob_test >= u).sum() <= 400]
revisar(lambda: np.isclose(umbral_elegido, min(_ok)))

---
# Parte 4. Publicar en Hugging Face Spaces

> **Definición. Hugging Face:** plataforma para compartir modelos, datasets y apps de IA.
>
> **Definición. Space:** una app alojada gratis en Hugging Face, con un link permanente del tipo `huggingface.co/spaces/usuario/nombre`. Funciona aunque cierren Colab.

Un Space de Gradio necesita tres archivos, que ya están en `space_fuga/`:

| Archivo | Qué contiene |
|---|---|
| `app.py` | El código de la app. Hugging Face lo ejecuta al iniciar. |
| `requirements.txt` | Las librerías necesarias. Gradio no hace falta: el Space ya lo trae. |
| `modelo_fuga.pkl` | El modelo guardado. |

Hay dos formas de subirlos. Hagan la **A** primero; la **B** les servirá para actualizar el proyecto rápido.

## Forma A: desde la web (sin código)

1. Creen una cuenta en [huggingface.co/join](https://huggingface.co/join) y confirmen el correo.
2. Arriba a la derecha: su foto > **New Space**.
3. Completen:
   - **Space name:** por ejemplo `riesgo-fuga-clientes` (sin espacios ni tildes).
   - **License:** `mit` está bien para un proyecto educativo.
   - **SDK:** **Gradio** (plantilla *Blank*).
   - **Hardware:** *CPU basic · Free*.
   - **Visibility:** *Public*, para poder compartir el link.
4. Clic en **Create Space**.
5. En la pestaña **Files**, clic en **Add file > Upload files**, arrastren los tres archivos (descárguenlos desde el panel de archivos de Colab, carpeta `space_fuga`) y clic en **Commit changes to main**.
6. Vuelvan a la pestaña **App**. Verán *Building* durante unos minutos y luego *Running*. La app ya tiene link permanente.

## Forma B: desde Colab con código

### Paso 1. Crear un token de escritura

1. En Hugging Face: foto > **Settings** > **Access Tokens** > **Create new token**.
2. Tipo **Write**, nombre `colab`. Copien el token (empieza con `hf_`).
3. En Colab, abran el panel de **secretos** (ícono de llave), agreguen uno llamado `HF_TOKEN` con el token como valor y activen *Acceso del notebook*.

**El token es como una contraseña: nunca lo escriban directamente en el notebook ni lo suban a GitHub.**

### Paso 2. Iniciar sesión

In [ ]:
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))      # usa el secreto de Colab
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()                            # si no hay secreto, abre un cuadro para pegar el token

api = HfApi()
usuario = api.whoami()["name"]
print("Sesión iniciada como:", usuario)

### Paso 3. Crear el Space y subir la carpeta

`create_repo` crea el Space (con `exist_ok=True` no falla si ya existe) y `upload_folder` sube todo el contenido de `space_fuga/`. Para actualizar la app más adelante, basta con volver a ejecutar `upload_folder`.

In [ ]:
NOMBRE_SPACE = "riesgo-fuga-clientes"          # cámbialo si quieres
repo_id = f"{usuario}/{NOMBRE_SPACE}"

api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)
api.upload_folder(folder_path="space_fuga", repo_id=repo_id, repo_type="space",
                  commit_message="Primera versión de la app de riesgo de fuga")

print("Listo. Tu app quedará en:")
print(f"https://huggingface.co/spaces/{repo_id}")

### Paso 4. Revisar que funcione

Abran el link. Los primeros minutos dirá *Building*. Si aparece **Runtime error**, abran la pestaña **Logs**. Los errores más comunes:

| Error en los logs | Causa | Solución |
|---|---|---|
| `ModuleNotFoundError: No module named 'sklearn'` | Falta la librería en `requirements.txt` | Agregar `scikit-learn==<versión>` |
| Error al hacer `pickle.load` o `AttributeError` | Versión de scikit-learn distinta a la del entrenamiento | Usar en `requirements.txt` la misma versión que imprime `sklearn.__version__` en Colab |
| `FileNotFoundError: modelo_fuga.pkl` | El modelo no se subió o tiene otro nombre | Revisar la pestaña *Files* |
| La app abre pero predice raro | Orden de entradas distinto al de la función | Revisar que `inputs` siga el orden de los argumentos |

**Para compartir:** el link del Space va en el README de su repositorio de GitHub y en la presentación final del capstone.

---
# Parte 5. Desafío: su propio proceso completo

Repitan el proceso de la Parte 3 con un dataset de Kaggle elegido por ustedes, de forma individual o con su equipo del capstone, y publíquenlo en un Space.

**Pasos:**

1. Elijan un dataset en Kaggle (revisen usability, descripción y licencia).
2. Descárguenlo con `kagglehub` y hagan un EDA breve.
3. Definan la pregunta, la variable objetivo y el tipo de problema.
4. Preparen los datos y armen un `Pipeline`.
5. Comparen al menos tres modelos con validación cruzada y ajusten el mejor con `GridSearchCV`.
6. Evalúen en prueba con las métricas adecuadas a su problema.
7. Guarden el modelo y creen una app con `gr.Blocks`.
8. Publíquenla en Hugging Face y compartan el link en el chat del curso.

**Datasets sugeridos de Kaggle** (identificadores para `kagglehub`):

| Identificador | Problema | Tipo |
|---|---|---|
| `shrutimechlearn/churn-modelling` | Fuga de clientes de un banco | Clasificación |
| `uciml/adult-census-income` | ¿Ingreso anual sobre 50 mil dólares? | Clasificación |
| `uciml/red-wine-quality-cortez-et-al-2009` | Calidad de vinos tintos | Clasificación o regresión |
| `uciml/mushroom-classification` | ¿Hongo comestible o venenoso? | Clasificación |
| `mirichoi0218/insurance` | Costo de un seguro médico | Regresión |
| `camnugent/california-housing-prices` | Precio de viviendas | Regresión |

Si eligen un problema de **regresión**, cambien las métricas por MAE, RMSE y R², y en la app muestren un número con `gr.Number` en vez de `gr.Label`.

---
# Resumen de la clase

| Tema | Lo esencial |
|---|---|
| Gradio | Una app es una función con interfaz. El orden de `inputs` y `outputs` sigue el de la función. `Interface` para lo simple, `Blocks` para diseño propio. |
| Kaggle | Revisar usability, descripción y licencia. Descargar con `kagglehub` usando el identificador `usuario/dataset`. Token solo para competencias. |
| Proyecto | Con clases desbalanceadas, la exactitud engaña: mirar recall y ROC AUC, usar `stratify` y `class_weight`. El umbral es una decisión de negocio. |
| Hugging Face | Un Space de Gradio necesita `app.py`, `requirements.txt` y el modelo. Misma versión de scikit-learn que en el entrenamiento. Tokens en secretos, nunca en el código. |

Guarden el notebook en el repositorio del curso, en `ejercicios_estudiantes`, como `clase06_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1

In [ ]:
def convertir(monto_clp, moneda):
    return round(monto_clp / TASAS[moneda], 2)

app_conversor = gr.Interface(
    fn=convertir,
    inputs=[gr.Number(value=100_000, label="Monto en pesos (CLP)"), gr.Dropdown(list(TASAS), value="USD", label="Moneda")],
    outputs=gr.Number(label="Monto convertido"),
    title="Conversor de pesos chilenos",
    flagging_mode="never",
)
print(convertir(95000, "USD"), convertir(79000, "UF"))
# app_conversor.launch()

### Ejercicio 2

In [ ]:
def procesar_imagen(imagen, efecto):
    if imagen is None:
        return None
    img = imagen.astype(float)
    if efecto == "Escala de grises":
        gris = img @ np.array([0.299, 0.587, 0.114])
        img = np.stack([gris] * 3, axis=-1)
    elif efecto == "Negativo":
        img = 255 - img
    elif efecto == "Más brillo":
        img = np.clip(img * 1.4, 0, 255)
    elif efecto == "Espejo":
        img = img[:, ::-1]
    return img.astype(np.uint8)

print(np.array_equal(procesar_imagen(prueba, "Espejo"), prueba[:, ::-1]))
# Para la app, agrega "Espejo" a las opciones del gr.Radio y vuelve a crear app_imagen

### Ejercicio 3

In [ ]:
id_dataset = "uciml/red-wine-quality-cortez-et-al-2009"

### Ejercicio 4

In [ ]:
fuga_por_contrato = datos.groupby("Contract")["Churn"].mean().sort_values(ascending=False)
fuga_por_contrato

### Ejercicio 5

In [ ]:
umbral_elegido = None
for u in umbrales:                       # recorre de menor a mayor
    if (prob_test >= u).sum() <= 400:
        umbral_elegido = u               # el primero que cumple es el más bajo
        break
print("Umbral:", umbral_elegido, "| Clientes a contactar:", (prob_test >= umbral_elegido).sum(),
      "| Recall:", round(recall_score(y_test, prob_test >= umbral_elegido), 3))